# Introducción

**Modelos de Machine Learning para clasificación y regresión: optimización de hiperparámetros, optimización computacional y validación estadística**

| | |
|---|---|
| **Autor** | Donny Torres |
| **Programa** | Maestría en Analítica de Datos, Universidad del Norte (Barranquilla) |
| **Asignatura** | Machine Learning |
| **Fecha** | Octubre de 2026 |
| **Conjunto de datos** | *Bank Marketing* con contexto social y económico (`bank-additional-full.csv`) |
| **Fuente** | UCI Machine Learning Repository: <https://archive.ics.uci.edu/dataset/222/bank+marketing> (licencia CC BY 4.0) |
| **Tareas** | Clasificación de `y` (suscribe o no) y regresión de `log1p(duration)` (duración de la llamada), ambas con información previa a la llamada y validación cronológica |

Este cuaderno desarrolla el tercer entregable del proyecto final y continúa el trabajo del Entregable 2: mismo conjunto de datos, misma partición cronológica 80/20, mismo preprocesamiento y el mismo modelo base (regresión logística con C = 0,01) como referencia. Sobre esa base construye el *pipeline* combinatorio que pide la guía: 7 modelos de clasificación × 4 técnicas de balanceo × 4 métodos de optimización (112), 7 modelos de regresión × 4 métodos (28) y, como extensión propia, el método GOAT iterativo en los 5 modelos con tres o más hiperparámetros (25), para un total de **165 combinaciones** evaluadas con **validación cruzada anidada**.

**Organización.** Las secciones 1 a 3 conservan la descripción y la auditoría de calidad de la base de datos, la reserva del conjunto de prueba, el análisis exploratorio y el preprocesamiento del Entregable 2, que ya incorporan las correcciones de la primera entrega (componente temporal completo, regularización elegida por pérdida logarítmica, estabilidad de coeficientes y recalibración mensual del intercepto); la sección 3.8 agrega los hallazgos nuevos que motivan las decisiones de este entregable. Las referencias del tipo "enunciado, sección x" de esas secciones remiten a la guía del primer entregable. Las secciones nuevas siguen la guía técnica de este entregable (*Modelos de Machine Learning para Clasificación y Regresión*):

| Sección del cuaderno | Contenido | Sección de la guía |
|---|---|---|
| 4. Diseño experimental | Explosión combinatoria, validación anidada, métrica de validación y reproducibilidad | 2, 3.2, 5.1.1 y 7.3 |
| 5. Optimización de hiperparámetros | Espacios de búsqueda, Grid, Random, Bayesiana, Genética, GOAT iterativo, multi-fidelidad, curvas *anytime* y semillas | 3 |
| 6. Optimización computacional | Complejidad teórica frente a tiempos medidos, paralelización y perfilamiento | 4 |
| 7. Evaluación de modelos | Tabla maestra, balanceo, métricas en prueba, calibración, residuos, SHAP y LIME, robustez | 5 |
| 8. Comparación estadística | Friedman, Nemenyi y diagrama CD, DeLong, MCS, Giacomini-White, Clark-West, Diebold-Mariano, *bootstrap* y tamaño de efecto | 6 |
| 9 y 10 | Conclusiones, limitaciones y referencias | 9 |

**Ejecución.** Las 165 combinaciones con validación anidada implican más de 50 000 ajustes de modelos, así que se ejecutan con los *scripts* de la carpeta `scripts/` (en paralelo y con registro por tarea en `resultados/corridas/`) y este cuaderno lee sus resultados. Con `RECALCULAR = True` en la celda de configuración del proyecto el cuaderno vuelve a lanzar esos *scripts*. Todas las interpretaciones se generan con los valores calculados en la misma ejecución, de forma que las cifras del texto coinciden siempre con las salidas.

## Configuración del entorno y reproducibilidad

La partición entrenamiento-prueba es cronológica y, por tanto, determinista. Para todo procedimiento aleatorio se fija una **semilla global única** (`SEED = 42`), definida en `src/bm/config.py` y propagada de forma explícita a `numpy`, scikit-learn (`random_state` de cada estimador), XGBoost, SMOTE y ADASYN, Optuna (`TPESampler(seed=...)`) y DEAP (módulo `random`). El entorno es `bm_e3` (Python 3.12) y las versiones exactas de todas las librerías están congeladas en `requirements.txt`; la tabla siguiente reporta las del análisis exploratorio, y la celda de módulos del tercer entregable (antes de la sección 3.8), las que se agregan en este entregable.

In [1]:
import platform
import warnings
from pathlib import Path

import IPython
import jinja2
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
import statsmodels
from IPython.display import Markdown, display
from scipy import stats
from sklearn import set_config
from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import IsolationForest
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, brier_score_loss, confusion_matrix,
                             f1_score, precision_recall_curve, precision_score, recall_score, roc_auc_score,
                             roc_curve, silhouette_score)
from sklearn.model_selection import (GridSearchCV, StratifiedKFold, TimeSeriesSplit, TunedThresholdClassifierCV,
                                     cross_val_predict, cross_validate)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor

SEED = 42
np.random.seed(SEED)
set_config(display="text")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

# Paleta validada (azul y naranja son distinguibles con daltonismo); tinta y rejilla recesivas
AZUL, NARANJA, AGUAMARINA, AMARILLO, ROJO = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e34948"
TINTA, TINTA_2, GRIS, REJILLA, FONDO = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#fcfcfb"
COLOR_Y = {0: AZUL, 1: NARANJA}
ETIQUETA_Y = {0: "no suscribe", 1: "suscribe"}
CMAP_SEQ = mpl.colors.LinearSegmentedColormap.from_list("secuencial", ["#f0efec", "#9ec5f4", "#2a78d6", "#0d366b"])
CMAP_DIV = mpl.colors.LinearSegmentedColormap.from_list("divergente", [AZUL, "#f0efec", ROJO])
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 110, "figure.facecolor": FONDO, "axes.facecolor": FONDO,
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": TINTA_2, "axes.titlecolor": TINTA, "axes.titlesize": 10.5,
    "axes.titleweight": "bold", "axes.labelsize": 9, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": REJILLA, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "xtick.color": GRIS, "ytick.color": GRIS, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "legend.frameon": False, "legend.fontsize": 8, "figure.titlesize": 12, "figure.titleweight": "bold",
    "axes.prop_cycle": mpl.cycler(color=[AZUL, NARANJA, AGUAMARINA, AMARILLO]), "lines.linewidth": 2,
})


def num(x, d=0):
    """Número con coma decimal y espacio de miles."""
    return f"{x:,.{d}f}".replace(",", " ").replace(".", ",")


def num_c(x):
    """Valor de un hiperparámetro sin ceros sobrantes (0,001; 0,1; 100)."""
    return f"{x:g}".replace(".", ",")


def pct(x, d=1):
    return f"{num(100 * x, d)} %"


def pval(p):
    return "< 0,001" if p < 0.001 else num(p, 3)


def p_txt(p):
    return "p < 0,001" if p < 0.001 else f"p = {num(p, 3)}"


def interpretar(interpretacion, implicacion, accion):
    display(Markdown(f"**Interpretación.** {interpretacion}\n\n{implicacion}\n\n{accion}"))


def conectar(conector, texto):
    """Antepone un conector al texto y pasa a minúscula su primera letra."""
    return f"{conector} {texto[:1].lower()}{texto[1:]}" if texto else texto


def tabla(df, dec=3, indice=True):
    sty = df.style.format(precision=dec, decimal=",", thousands=" ", na_rep="—")
    if not indice:
        sty = sty.hide(axis="index")
    display(sty)


def mapa_calor(ax, M, filas, columnas, cmap, vmin, vmax, dec=2, tam=7):
    M = np.asarray(M, dtype=float)
    im = ax.imshow(M, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
    ax.set_xticks(range(len(columnas)), columnas, rotation=45, ha="right")
    ax.set_yticks(range(len(filas)), filas)
    ax.grid(False)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            if np.isfinite(M[i, j]):
                rel = (M[i, j] - vmin) / (vmax - vmin)
                oscuro = abs(rel - 0.5) > 0.35 if cmap is CMAP_DIV else rel > 0.6
                ax.text(j, i, num(M[i, j], dec), ha="center", va="center", fontsize=tam,
                        color="white" if oscuro else TINTA)
    return im


def enumerar(items, codigo=True):
    """Une una lista en español: a, b y c (con formato de código si se pide)."""
    items = [f"`{i}`" if codigo else str(i) for i in items]
    if not items:
        return ""
    return items[0] if len(items) == 1 else ", ".join(items[:-1]) + " y " + items[-1]


def enumerar(items, codigo=True):
    """Une una lista en español: a, b y c (con formato de código si se pide)."""
    items = [f"`{i}`" if codigo else str(i) for i in items]
    if not items:
        return ""
    return items[0] if len(items) == 1 else ", ".join(items[:-1]) + " y " + items[-1]


def magnitud(valor, cortes, nombres=("despreciable", "pequeño", "mediano", "grande")):
    """Etiqueta orientativa del tamaño de efecto según cortes convencionales."""
    for corte, nombre in zip(cortes, nombres):
        if abs(valor) < corte:
            return nombre
    return nombres[-1]


versiones = pd.DataFrame({
    "Componente": ["Python", "numpy", "pandas", "scipy", "scikit-learn", "statsmodels", "matplotlib", "jinja2",
                   "IPython"],
    "Versión": [platform.python_version(), np.__version__, pd.__version__, scipy.__version__, sklearn.__version__,
                statsmodels.__version__, mpl.__version__, jinja2.__version__, IPython.__version__]})
tabla(versiones, indice=False)
print(f"Semilla global: SEED = {SEED}")

Componente,Versión
Python,3.12.14
numpy,2.5.4
pandas,3.0.6
scipy,1.18.1
scikit-learn,1.9.1
statsmodels,0.15.0
matplotlib,3.11.2
jinja2,3.1.6
IPython,9.17.1


Semilla global: SEED = 42
